In [2]:
class Money:
    def __init__(self, rubles = 0, kopecks = 0, currency = "BYN"):
        self.rubles = rubles
        self.kopecks = kopecks
        self.currency = currency
        self.total_kop = rubles * 100 + kopecks

    rates = {
        ("USD", "BYN"): 3.27,
        ("BYN", "USD"): 0.31,
        ("EUR", "BYN"): 3.55,
        ("BYN", "EUR"): 0.28,
        ("USD", "EUR"): 0.92,
        ("EUR", "USD"): 1.09,
    }

    def convert(self, new_currency):
      if self.currency == new_currency:
          return
      rate = Money.rates[(self.currency, new_currency)]
      self.total_kop = int(self.total_kop * rate)
      self.rubles = self.total_kop // 100
      self.kopecks = self.total_kop % 100
      self.currency = new_currency

    def write(self):
        return f"{self.rubles},{self.kopecks:2} {self.currency}"

    def add(self, other, n: float):
        if self.currency != other.currency:
            return
        self.total_kop += n
        self.rubles = self.total_kop // 100
        self.kopecks = self.total_kop % 100
        other.total_kop -= n
        other.rubles = other.total_kop // 100
        other.kopecks = other.total_kop % 100

    def sub(self, other, n: float):
        if self.currency != other.currency:
            return
        self.total_kop -= n
        self.rubles = self.total_kop // 100
        self.kopecks = self.total_kop % 100
        other.total_kop += n
        other.rubles = other.total_kop // 100
        other.kopecks = other.total_kop % 100

    def mul(self, n: float):
        total = self.total_kop * n
        rub = total // 100
        kop = total % 100
        self.rubles = rub
        self.kopecks = kop

    def div(self, n: float):
        if n == 0:
            print ("Error")
            return
        total = self.total_kop / n
        rub = total // 100
        kop = total % 100
        self.rubles = int(rub)
        self.kopecks = int(kop)

    def equal(self, other):
        if not isinstance(other, Money):
            return False
        if self.currency == other.currency and self.total_kop == other.total_kop:
            return True
        else:
            return False

In [3]:
class BYN(Money):
    def __init__(self, rubles=0, kopecks=0):
        super().__init__(rubles, kopecks, currency="BYN")

In [4]:
class USD(Money):
    def __init__(self, rubles=0, kopecks=0):
        super().__init__(rubles, kopecks, currency="USD")

In [5]:
class EUR(Money):
    def __init__(self, rubles=0, kopecks=0):
        super().__init__(rubles, kopecks, currency="EUR")

In [6]:
class Account:
    def __init__(self, name):
        self.name = name
        self.money = []

    def deposit(self, money):
      for m in self.money:
        if m.currency == money.currency:
            m.total_kop += money.total_kop
            m.rubles = m.total_kop // 100
            m.kopecks = m.total_kop % 100
            return
      self.money.append(money)

    def withdraw(self, money):
        for m in self.money:
            if m.currency == money.currency:
                if m.total_kop >= money.total_kop:
                    m.total_kop -= money.total_kop
                    m.rubles = m.total_kop // 100
                    m.kopecks = m.total_kop % 100
                    return True
                else:
                    return False
        return False

    def get_balance(self, currency):
        for m in self.money:
            if m.currency == currency:
                return m.write()
        return "Пусто"

    def transfer(self, other_account, money):
        if self.withdraw(money):
            other_account.deposit(money)
            return True
        print("Недостаточно средств для перевода")
        return False


In [7]:
class BankAccount:
    def __init__(self, owner):
        self.owner = owner
        self.accounts = []

    def add_account(self, account):
      self.accounts.append(account)

    def remove_account(self, name):
        for acc in self.accounts:
            if acc.name == name:
                self.accounts.remove(acc)
                return

    def total_balance(self, currency):
        total = 0
        for acc in self.accounts:
            for m in acc.money:
                if m.currency == currency:
                    total += m.total_kop
        return Money(total // 100, total % 100, currency)

    def find_account(self, name):
        for acc in self.accounts:
            if acc.name == name:
                return acc
        return None

    def transfer_between(self, from_name, to_name, money):
      from_acc = self.find_account(from_name)
      to_acc = self.find_account(to_name)
      if from_acc is None or to_acc is None:
        print("Счёт не найден")
        return
      from_acc.transfer(to_acc, money)


In [8]:
hundred_byn = BYN(100, 50)
fifty_usd = USD(50, 0)
twenty_eur = EUR(20, 0)

print(hundred_byn.write())

100,50 BYN


In [9]:
savings = Account("Сбережения")
print(savings.name)
print(savings.money)

Сбережения
[]


In [10]:
print(savings.get_balance("BYN"))
print(savings.get_balance("EUR"))

Пусто
Пусто


In [17]:
result = savings.withdraw(BYN(50, 0))
print(result)
print(savings.get_balance("BYN"))

result = savings.withdraw(BYN(1000, 0))
print(result)

True
450, 0 BYN
False


In [12]:
daniel = BankAccount("Daniel")

payday = Account("Зарплата")

daniel.add_account(savings)
daniel.add_account(payday)

print(daniel.owner)
print(len(daniel.accounts))
print(daniel.accounts)

Daniel
2
[<__main__.Account object at 0x7b2083c25010>, <__main__.Account object at 0x7b2083c92ea0>]


In [13]:
payday.deposit(BYN(1000, 0))

daniel.transfer_between("Зарплата", "Сбережения", BYN(500, 0))

print(payday.get_balance("BYN"))
print(savings.get_balance("BYN"))

500, 0 BYN
500, 0 BYN


In [14]:
total = daniel.total_balance("BYN")
print(total.write())

1000, 0 BYN


In [19]:
my_money = USD(100, 0)
print(my_money.write())

my_money.convert("BYN")
print(my_money.write())

100, 0 USD
327, 0 BYN


In [16]:
daniel.transfer_between("Зарплатный", "Сбережения", BYN(500, 0))

Счёт не найден
